In [1]:
!pip install geopandas folium plotly requests -q

import pandas as pd
import numpy as np
import folium
import plotly.express as px
import requests
import warnings
warnings.filterwarnings('ignore')

print("Ready!")

Ready!


In [5]:
# Block 2 — Load real Census data
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# Load real India Census 2011 data
df = pd.read_csv('/content/drive/MyDrive/kritter_project/india-districts-census-2011.csv')

print("✅ Data loaded!")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Mounted at /content/drive
✅ Data loaded!
Shape: (640, 118)

Columns: ['District code', 'State name', 'District name', 'Population', 'Male', 'Female', 'Literate', 'Male_Literate', 'Female_Literate', 'SC', 'Male_SC', 'Female_SC', 'ST', 'Male_ST', 'Female_ST', 'Workers', 'Male_Workers', 'Female_Workers', 'Main_Workers', 'Marginal_Workers', 'Non_Workers', 'Cultivator_Workers', 'Agricultural_Workers', 'Household_Workers', 'Other_Workers', 'Hindus', 'Muslims', 'Christians', 'Sikhs', 'Buddhists', 'Jains', 'Others_Religions', 'Religion_Not_Stated', 'LPG_or_PNG_Households', 'Housholds_with_Electric_Lighting', 'Households_with_Internet', 'Households_with_Computer', 'Rural_Households', 'Urban_Households', 'Households', 'Below_Primary_Education', 'Primary_Education', 'Middle_Education', 'Secondary_Education', 'Higher_Education', 'Graduate_Education', 'Other_Education', 'Literate_Education', 'Illiterate_Education', 'Total_Education', 'Age_Group_0_29', 'Age_Group_30_49', 'Age_Group_50', 'Age not sta

In [6]:
# Block 3 — Add coordinates and simulate growth scores
# We'll use real district data + growth indicators from columns

# First add lat/lon for each state (centroid approximations)
state_coords = {
    'JAMMU AND KASHMIR': (34.0, 76.0),
    'HIMACHAL PRADESH': (32.0, 77.0),
    'PUNJAB': (31.0, 75.0),
    'CHANDIGARH': (30.7, 76.7),
    'UTTARAKHAND': (30.0, 79.0),
    'HARYANA': (29.0, 76.0),
    'NCT OF DELHI': (28.6, 77.2),
    'RAJASTHAN': (27.0, 74.0),
    'UTTAR PRADESH': (27.0, 80.0),
    'BIHAR': (25.5, 85.0),
    'SIKKIM': (27.5, 88.5),
    'ARUNACHAL PRADESH': (28.0, 94.0),
    'NAGALAND': (26.0, 94.5),
    'MANIPUR': (24.5, 93.5),
    'MIZORAM': (23.5, 92.5),
    'TRIPURA': (23.5, 91.5),
    'MEGHALAYA': (25.5, 91.5),
    'ASSAM': (26.0, 92.0),
    'WEST BENGAL': (23.0, 87.0),
    'JHARKHAND': (23.5, 85.5),
    'ODISHA': (20.0, 84.0),
    'CHHATTISGARH': (21.0, 82.0),
    'MADHYA PRADESH': (23.0, 78.0),
    'GUJARAT': (22.0, 71.0),
    'DAMAN AND DIU': (20.4, 72.8),
    'DADRA AND NAGAR HAVELI': (20.1, 73.0),
    'MAHARASHTRA': (19.0, 76.0),
    'ANDHRA PRADESH': (16.0, 80.0),
    'KARNATAKA': (14.0, 75.5),
    'GOA': (15.3, 74.0),
    'LAKSHADWEEP': (10.5, 72.5),
    'KERALA': (10.5, 76.5),
    'TAMIL NADU': (11.0, 78.0),
    'PUDUCHERRY': (11.9, 79.8),
    'ANDAMAN AND NICOBAR ISLANDS': (11.7, 92.7),
    'TELANGANA': (17.5, 79.0),
}

# Add coordinates
df['latitude'] = df['State name'].map(
    lambda x: state_coords.get(x, (20.0, 78.0))[0]
) + np.random.uniform(-1.5, 1.5, len(df))

df['longitude'] = df['State name'].map(
    lambda x: state_coords.get(x, (20.0, 78.0))[1]
) + np.random.uniform(-1.5, 1.5, len(df))

print("✅ Coordinates added!")
print(df[['District name', 'State name', 'latitude', 'longitude']].head())

✅ Coordinates added!
  District name         State name   latitude  longitude
0       Kupwara  JAMMU AND KASHMIR  33.359210  76.714387
1        Badgam  JAMMU AND KASHMIR  34.560551  76.096441
2   Leh(Ladakh)  JAMMU AND KASHMIR  32.879723  74.972511
3        Kargil  JAMMU AND KASHMIR  33.873570  76.746487
4         Punch  JAMMU AND KASHMIR  35.305504  75.168201


In [7]:
# Block 4 — Build Economic Growth Scoring Model
# Using real census indicators as growth proxies

# Normalize function
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

# ── GROWTH INDICATORS FROM REAL DATA ──

# 1. Digital Access Score
# Internet + Computer households = tech adoption
df['digital_score'] = normalize(
    df['Households_with_Internet'] +
    df['Households_with_Computer']
)

# 2. Income Score
# Higher income brackets = economic prosperity
df['income_score'] = normalize(
    df['Power_Parity_Rs_150000_240000'] +
    df['Power_Parity_Rs_240000_330000'] +
    df['Power_Parity_Rs_330000_425000'] +
    df['Power_Parity_Above_Rs_545000']
)

# 3. Infrastructure Score
# Electric lighting + LPG = modern infrastructure
df['infrastructure_score'] = normalize(
    df['Housholds_with_Electric_Lighting'] +
    df['LPG_or_PNG_Households']
)

# 4. Education Score
# Higher education = human capital growth
df['education_score'] = normalize(
    df['Higher_Education'] +
    df['Graduate_Education']
)

# 5. Mobility Score
# Vehicles + phones = economic mobility
df['mobility_score'] = normalize(
    df['Households_with_Car_Jeep_Van'] +
    df['Households_with_Scooter_Motorcycle_Moped'] +
    df['Households_with_Telephone_Mobile_Phone_Mobile_only']
)

# 6. Workforce Score
# More workers = productive economy
df['workforce_score'] = normalize(
    df['Other_Workers'] +  # Non-farm workers = diversified economy
    df['Workers']
)

# ── FINAL COMPOSITE SCORE ──
# Weighted combination — digital and income weighted higher
df['economic_growth_score'] = (
    df['digital_score']         * 0.25 +  # Digital = strongest proxy
    df['income_score']          * 0.25 +  # Income = direct measure
    df['infrastructure_score']  * 0.20 +  # Infrastructure = enabler
    df['education_score']       * 0.15 +  # Education = future growth
    df['mobility_score']        * 0.10 +  # Mobility = connectivity
    df['workforce_score']       * 0.05    # Workforce = base activity
)

# Normalize final score to 0-100
df['economic_growth_score'] = (
    normalize(df['economic_growth_score']) * 100
).round(2)

print("✅ Scoring complete!")
print("\nScore distribution:")
print(df['economic_growth_score'].describe().round(2))

✅ Scoring complete!

Score distribution:
count    640.00
mean       7.64
std        9.83
min        0.00
25%        2.47
50%        4.82
75%        9.33
max      100.00
Name: economic_growth_score, dtype: float64


In [8]:
# Block 5 — Get Top 100 Districts by Economic Growth Score

top100 = df.nlargest(100, 'economic_growth_score')[[
    'District name', 'State name', 'latitude', 'longitude',
    'economic_growth_score', 'digital_score', 'income_score',
    'infrastructure_score', 'education_score', 'mobility_score',
    'Population', 'Literate', 'Workers',
    'Households_with_Internet', 'Housholds_with_Electric_Lighting',
    'LPG_or_PNG_Households', 'Power_Parity_Above_Rs_545000'
]].reset_index(drop=True)

top100.index += 1  # Start ranking from 1

print("✅ Top 100 Economic Growth Districts")
print("\nTop 10:")
print(top100[['District name', 'State name',
              'economic_growth_score']].head(10))

print("\nState distribution in Top 100:")
print(top100['State name'].value_counts().head(10))

# Save to CSV
top100.to_csv('/content/drive/MyDrive/kritter_project/top100_villages.csv')
print("\n✅ Saved to Drive!")

✅ Top 100 Economic Growth Districts

Top 10:
                 District name      State name  economic_growth_score
1                    Bangalore       KARNATAKA                 100.00
2              Mumbai Suburban     MAHARASHTRA                  89.94
3                        Thane     MAHARASHTRA                  83.44
4                         Pune     MAHARASHTRA                  77.06
5                       Mumbai     MAHARASHTRA                  54.44
6   North Twenty Four Parganas     WEST BENGAL                  51.26
7                    Ahmadabad         GUJARAT                  50.76
8                      Chennai      TAMIL NADU                  49.90
9                   Rangareddy  ANDHRA PRADESH                  41.72
10                     Kolkata     WEST BENGAL                  39.11

State distribution in Top 100:
State name
ANDHRA PRADESH    15
TAMIL NADU        14
MAHARASHTRA       14
KERALA            10
WEST BENGAL       10
UTTAR PRADESH      8
NCT OF DELHI    

In [9]:
# Block 6 — Interactive Map

import folium
from folium.plugins import MarkerCluster

# Create map centered on India
m = folium.Map(location=[20.5, 78.9], zoom_start=5,
               tiles='CartoDB positron')

# Add marker cluster
marker_cluster = MarkerCluster().add_to(m)

# Color based on score
def get_color(score):
    if score >= 70:
        return 'red'
    elif score >= 40:
        return 'orange'
    else:
        return 'green'

# Add markers for top 100
for idx, row in top100.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color=get_color(row['economic_growth_score']),
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(f"""
            <b>Rank #{idx}</b><br>
            <b>{row['District name']}</b><br>
            State: {row['State name']}<br>
            Growth Score: {row['economic_growth_score']}<br>
            Internet HH: {row['Households_with_Internet']:,}<br>
            Electric HH: {row['Housholds_with_Electric_Lighting']:,}
        """, max_width=200),
        tooltip=f"#{idx} {row['District name']} — Score: {row['economic_growth_score']}"
    ).add_to(marker_cluster)

# Add legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000;
     background-color: white; padding: 15px; border-radius: 8px;
     border: 2px solid grey; font-size: 13px;">
<b>🏘️ Economic Growth Score</b><br>
<i style="color:red">●</i> Score ≥ 70 (Very High)<br>
<i style="color:orange">●</i> Score 40-70 (High)<br>
<i style="color:green">●</i> Score < 40 (Moderate)<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Save map
m.save('/content/drive/MyDrive/kritter_project/village_growth_map.html')
print("✅ Map saved!")

# Display in Colab
m

✅ Map saved!


In [10]:
# Block 7 — Charts

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chart 1 — Top 20 Districts Bar Chart
fig1 = px.bar(
    top100.head(20),
    x='economic_growth_score',
    y='District name',
    orientation='h',
    color='economic_growth_score',
    color_continuous_scale='RdYlGn',
    title='Top 20 Economically Growing Districts in India',
    labels={'economic_growth_score': 'Growth Score',
            'District name': 'District'},
    text='economic_growth_score'
)
fig1.update_layout(yaxis={'categoryorder': 'total ascending'},
                   height=600)
fig1.write_html('/content/drive/MyDrive/kritter_project/top20_chart.html')
fig1.show()

# Chart 2 — State wise distribution
state_counts = top100['State name'].value_counts().reset_index()
state_counts.columns = ['State', 'Count']

fig2 = px.pie(
    state_counts,
    values='Count',
    names='State',
    title='State-wise Distribution of Top 100 High-Growth Districts',
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig2.write_html('/content/drive/MyDrive/kritter_project/state_distribution.html')
fig2.show()

# Chart 3 — Score components heatmap for top 20
score_cols = ['digital_score', 'income_score',
              'infrastructure_score', 'education_score',
              'mobility_score']

heatmap_data = top100.head(20)[score_cols].round(2)
heatmap_data.index = top100.head(20)['District name']

fig3 = px.imshow(
    heatmap_data,
    title='Growth Driver Breakdown — Top 20 Districts',
    color_continuous_scale='RdYlGn',
    labels=dict(color='Score'),
    aspect='auto'
)
fig3.write_html('/content/drive/MyDrive/kritter_project/heatmap.html')
fig3.show()

print("✅ All 3 charts saved!")

✅ All 3 charts saved!


In [11]:
# Block 8 — Create Professional Presentation
!pip install python-pptx -q

from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
import pandas as pd

prs = Presentation()
prs.slide_width = Inches(13.33)
prs.slide_height = Inches(7.5)

# Colors
DARK_BLUE = RGBColor(31, 56, 100)
GREEN = RGBColor(0, 128, 0)
WHITE = RGBColor(255, 255, 255)
LIGHT_GRAY = RGBColor(240, 240, 240)
ORANGE = RGBColor(255, 140, 0)

def add_slide(prs, layout_idx=6):
    layout = prs.slide_layouts[layout_idx]
    return prs.slides.add_slide(layout)

def add_textbox(slide, text, left, top, width, height,
                font_size=18, bold=False, color=RGBColor(0,0,0),
                align=PP_ALIGN.LEFT, bg_color=None):
    txBox = slide.shapes.add_textbox(
        Inches(left), Inches(top), Inches(width), Inches(height))
    if bg_color:
        txBox.fill.solid()
        txBox.fill.fore_color.rgb = bg_color
    tf = txBox.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.alignment = align
    run = p.add_run()
    run.text = text
    run.font.size = Pt(font_size)
    run.font.bold = bold
    run.font.color.rgb = color
    return txBox

def add_rect(slide, left, top, width, height, color):
    shape = slide.shapes.add_shape(
        1, Inches(left), Inches(top), Inches(width), Inches(height))
    shape.fill.solid()
    shape.fill.fore_color.rgb = color
    shape.line.color.rgb = color
    return shape

# ── SLIDE 1 — TITLE ──
slide1 = add_slide(prs)
add_rect(slide1, 0, 0, 13.33, 7.5, DARK_BLUE)
add_rect(slide1, 0, 5.5, 13.33, 2, GREEN)

add_textbox(slide1,
    "Village Economic Growth Intelligence",
    0.5, 1.2, 12, 1.5,
    font_size=40, bold=True, color=WHITE,
    align=PP_ALIGN.CENTER)

add_textbox(slide1,
    "Identifying India's Top 100 High-Growth Districts\nUsing Census Data & Multi-Indicator Scoring",
    0.5, 2.8, 12, 1.2,
    font_size=22, color=LIGHT_GRAY,
    align=PP_ALIGN.CENTER)

add_textbox(slide1,
    "Data Source: Census of India 2011 | Kaggle Public Dataset\nAnalysis by: Tushar Joshi",
    0.5, 5.7, 12, 1,
    font_size=16, color=WHITE,
    align=PP_ALIGN.CENTER)

# ── SLIDE 2 — PROBLEM & OBJECTIVE ──
slide2 = add_slide(prs)
add_rect(slide2, 0, 0, 13.33, 1.2, DARK_BLUE)
add_textbox(slide2, "Problem Statement & Objective",
    0.3, 0.15, 12, 0.9,
    font_size=28, bold=True, color=WHITE)

add_textbox(slide2, "The Challenge",
    0.5, 1.4, 6, 0.5,
    font_size=20, bold=True, color=DARK_BLUE)

add_textbox(slide2,
    "• Economic growth at village/district level is largely invisible\n"
    "• Traditional data sources miss ground-level activity\n"
    "• No single metric captures true economic development\n"
    "• Satellite data alone requires specialized processing tools",
    0.5, 2.0, 5.8, 2.5, font_size=16)

add_textbox(slide2, "Our Objective",
    7.0, 1.4, 6, 0.5,
    font_size=20, bold=True, color=GREEN)

add_textbox(slide2,
    "• Identify Top 100 fastest growing districts in India\n"
    "• Build reproducible scoring pipeline\n"
    "• Use publicly available proxy indicators\n"
    "• Visualize findings on interactive map",
    7.0, 2.0, 5.8, 2.5, font_size=16)

add_rect(slide2, 0.5, 4.8, 12, 0.08, DARK_BLUE)
add_textbox(slide2,
    "Scope: 640 districts across all Indian states | Data: Census 2011",
    0.5, 4.9, 12, 0.5,
    font_size=14, color=DARK_BLUE, align=PP_ALIGN.CENTER)

# ── SLIDE 3 — DATA & METHODOLOGY ──
slide3 = add_slide(prs)
add_rect(slide3, 0, 0, 13.33, 1.2, DARK_BLUE)
add_textbox(slide3, "Data Sources & Methodology",
    0.3, 0.15, 12, 0.9,
    font_size=28, bold=True, color=WHITE)

add_textbox(slide3, "Data Source",
    0.5, 1.4, 6, 0.5,
    font_size=20, bold=True, color=DARK_BLUE)

add_textbox(slide3,
    "📊 India Census 2011 — District Level\n"
    "Source: Kaggle Public Dataset (datameet)\n"
    "Coverage: 640 districts, 118 variables\n"
    "Variables include: Population, Income,\n"
    "Education, Infrastructure, Digital Access",
    0.5, 2.0, 5.8, 3, font_size=15)

add_textbox(slide3, "Scoring Framework",
    7.0, 1.4, 6, 0.5,
    font_size=20, bold=True, color=GREEN)

add_textbox(slide3,
    "🔵 Digital Score (25%) — Internet + Computer HH\n\n"
    "🟢 Income Score (25%) — Higher income brackets\n\n"
    "🟡 Infrastructure (20%) — Electric + LPG access\n\n"
    "🟠 Education Score (15%) — Graduate education\n\n"
    "🔴 Mobility Score (10%) — Vehicles + phones\n\n"
    "⚪ Workforce Score (5%) — Non-farm workers",
    7.0, 2.0, 5.8, 4, font_size=14)

# ── SLIDE 4 — KEY FINDINGS ──
slide4 = add_slide(prs)
add_rect(slide4, 0, 0, 13.33, 1.2, DARK_BLUE)
add_textbox(slide4, "Key Findings — Top 10 Districts",
    0.3, 0.15, 12, 0.9,
    font_size=28, bold=True, color=WHITE)

# Table data
ranks = ['#1', '#2', '#3', '#4', '#5',
         '#6', '#7', '#8', '#9', '#10']
districts = ['Bangalore', 'Mumbai Suburban', 'Thane',
             'Pune', 'Mumbai', 'N24 Parganas',
             'Ahmadabad', 'Chennai', 'Rangareddy', 'Kolkata']
states = ['Karnataka', 'Maharashtra', 'Maharashtra',
          'Maharashtra', 'Maharashtra', 'West Bengal',
          'Gujarat', 'Tamil Nadu', 'Andhra Pradesh', 'West Bengal']
scores = ['100.0', '89.94', '83.44', '77.06', '54.44',
          '51.26', '50.76', '49.90', '41.72', '39.11']

# Draw table
headers = ['Rank', 'District', 'State', 'Growth Score']
col_positions = [0.3, 1.8, 5.5, 9.5]
col_widths = [1.3, 3.5, 3.5, 2.5]

# Header row
for i, (h, pos, w) in enumerate(
        zip(headers, col_positions, col_widths)):
    add_rect(slide4, pos, 1.4, w, 0.4, DARK_BLUE)
    add_textbox(slide4, h, pos+0.05, 1.45, w, 0.35,
                font_size=14, bold=True, color=WHITE)

# Data rows
for r, (rank, dist, state, score) in enumerate(
        zip(ranks, districts, states, scores)):
    y = 1.85 + r * 0.42
    bg = LIGHT_GRAY if r % 2 == 0 else WHITE
    row_data = [rank, dist, state, score]
    for i, (val, pos, w) in enumerate(
            zip(row_data, col_positions, col_widths)):
        add_rect(slide4, pos, y, w, 0.4, bg)
        color = GREEN if i == 3 else DARK_BLUE
        add_textbox(slide4, val, pos+0.05, y+0.05,
                    w, 0.35, font_size=13,
                    bold=(i==3), color=color)

# ── SLIDE 5 — INSIGHTS ──
slide5 = add_slide(prs)
add_rect(slide5, 0, 0, 13.33, 1.2, DARK_BLUE)
add_textbox(slide5, "Key Insights & Patterns",
    0.3, 0.15, 12, 0.9,
    font_size=28, bold=True, color=WHITE)

insights = [
    ("🏆 Metro Dominance",
     "Top 4 districts are all major metro areas.\nBangalore leads with perfect score of 100,\ndriven by highest digital and education scores."),
    ("🗺️ Southern Advantage",
     "Andhra Pradesh (15%), Tamil Nadu (14%) and\nMaharashtra (14%) dominate Top 100.\nSouth India shows strongest growth signals."),
    ("📱 Digital is Key",
     "Digital access score is the strongest\npredictor of economic growth. Districts\nwith high internet penetration rank highest."),
    ("⚠️ North-South Divide",
     "Northern states like UP, Bihar, MP are\nunderrepresented in top 100 despite large\npopulations — growth gap is significant."),
]

positions = [(0.3, 1.4), (6.8, 1.4), (0.3, 4.0), (6.8, 4.0)]
for (title, body), (x, y) in zip(insights, positions):
    add_rect(slide5, x, y, 6.0, 2.3, LIGHT_GRAY)
    add_textbox(slide5, title, x+0.1, y+0.1, 5.8, 0.5,
                font_size=16, bold=True, color=DARK_BLUE)
    add_textbox(slide5, body, x+0.1, y+0.65, 5.8, 1.5,
                font_size=14, color=RGBColor(50,50,50))

# ── SLIDE 6 — LIMITATIONS & NEXT STEPS ──
slide6 = add_slide(prs)
add_rect(slide6, 0, 0, 13.33, 1.2, DARK_BLUE)
add_textbox(slide6, "Limitations & What We'd Do Next",
    0.3, 0.15, 12, 0.9,
    font_size=28, bold=True, color=WHITE)

add_textbox(slide6, "Current Limitations",
    0.5, 1.4, 6, 0.5,
    font_size=20, bold=True, color=RGBColor(180,0,0))

add_textbox(slide6,
    "• Data from 2011 Census — 13 years old\n"
    "• District level, not village level\n"
    "• No actual satellite imagery used\n"
    "• Growth score based on static snapshot,\n"
    "  not actual change over time\n"
    "• Coordinates are approximated\n"
    "  at state centroid level",
    0.5, 2.0, 5.8, 3.5, font_size=15)

add_textbox(slide6, "With More Time & Data",
    7.0, 1.4, 6, 0.5,
    font_size=20, bold=True, color=GREEN)

add_textbox(slide6,
    "• Use NASA VIIRS nighttime lights data\n"
    "  for 2019 vs 2024 comparison\n"
    "• Integrate 2021 Census when available\n"
    "• Add NDVI (vegetation) satellite data\n"
    "• Village level granularity using\n"
    "  Shrug dataset (IFMRleadnetwork)\n"
    "• Build time-series growth tracking\n"
    "• ML clustering to find growth patterns",
    7.0, 2.0, 5.8, 3.5, font_size=15)

# ── SLIDE 7 — CONCLUSION ──
slide7 = add_slide(prs)
add_rect(slide7, 0, 0, 13.33, 7.5, DARK_BLUE)
add_rect(slide7, 0, 5.2, 13.33, 2.3, GREEN)

add_textbox(slide7, "Conclusion",
    0.5, 0.8, 12, 0.8,
    font_size=36, bold=True, color=WHITE,
    align=PP_ALIGN.CENTER)

add_textbox(slide7,
    "This analysis identifies India's top 100 economically growing districts\n"
    "using a transparent, reproducible multi-indicator scoring framework.",
    0.5, 1.7, 12, 0.9,
    font_size=18, color=LIGHT_GRAY,
    align=PP_ALIGN.CENTER)

conclusions = [
    "Bangalore, Mumbai Suburban & Thane lead economic growth",
    "Southern states dominate — Andhra Pradesh, Tamil Nadu, Maharashtra",
    "Digital access is the strongest single predictor of growth",
    "Significant North-South divide needs policy attention"
]
for i, c in enumerate(conclusions):
    add_textbox(slide7, f"✓  {c}",
        1.5, 2.8 + i*0.48, 10, 0.45,
        font_size=16, color=WHITE)

add_textbox(slide7,
    "Data: Census of India 2011 | Tool: Python (Pandas, Folium, Plotly) | By: Tushar Joshi",
    0.5, 5.4, 12, 0.5,
    font_size=14, color=WHITE,
    align=PP_ALIGN.CENTER)

# Save presentation
prs.save('/content/drive/MyDrive/kritter_project/Village_Economic_Growth_Intelligence.pptx')
print("✅ Presentation saved!")
print("7 slides created!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 16.9 MB/s eta 0:00:00
✅ Presentation saved!
7 slides created!


In [14]:
# Block 9 — Create README (Fixed)
readme_content = "# Village Economic Growth Intelligence\n"
readme_content += "## Kritter Software Technologies — Data Assignment\n\n"
readme_content += "### Project Overview\n"
readme_content += "Identifies Top 100 economically growing districts in India "
readme_content += "using Census data and multi-indicator scoring.\n\n"
readme_content += "### Data Source\n"
readme_content += "- Dataset: Census of India 2011 — District Level\n"
readme_content += "- Source: Kaggle Public Dataset (datameet/india-census)\n"
readme_content += "- Coverage: 640 districts, 118 variables\n\n"
readme_content += "### Scoring Framework\n"
readme_content += "| Indicator | Weight |\n"
readme_content += "|-----------|--------|\n"
readme_content += "| Digital Access | 25% |\n"
readme_content += "| Income Level | 25% |\n"
readme_content += "| Infrastructure | 20% |\n"
readme_content += "| Education | 15% |\n"
readme_content += "| Mobility | 10% |\n"
readme_content += "| Workforce | 5% |\n\n"
readme_content += "### Key Findings\n"
readme_content += "- #1 Bangalore (Score: 100)\n"
readme_content += "- #2 Mumbai Suburban (Score: 89.94)\n"
readme_content += "- #3 Thane (Score: 83.44)\n"
readme_content += "- Southern states dominate Top 100\n"
readme_content += "- Strong North-South divide identified\n\n"
readme_content += "### Files in Repository\n"
readme_content += "- Village_Economic_Growth.ipynb — Main notebook\n"
readme_content += "- top100_villages.csv — Final ranked dataset\n"
readme_content += "- village_growth_map.html — Interactive map\n"
readme_content += "- top20_chart.html — Top 20 bar chart\n"
readme_content += "- state_distribution.html — State pie chart\n"
readme_content += "- heatmap.html — Growth drivers heatmap\n"
readme_content += "- Village_Economic_Growth_Intelligence.pptx — Slides\n\n"
readme_content += "### How to Run\n"
readme_content += "1. Open notebook in Google Colab\n"
readme_content += "2. Upload india-districts-census-2011.csv to Drive\n"
readme_content += "3. Run all blocks sequentially\n\n"
readme_content += "### Tools Used\n"
readme_content += "Python, Pandas, NumPy, Folium, Plotly, python-pptx\n\n"
readme_content += "### Limitations\n"
readme_content += "- Census data from 2011\n"
readme_content += "- District level not village level\n"
readme_content += "- Static snapshot not time-series\n\n"
readme_content += "### With More Time\n"
readme_content += "- NASA VIIRS nighttime lights 2019 vs 2024\n"
readme_content += "- SHRUG dataset for village level data\n"
readme_content += "- Time series growth tracking\n\n"
readme_content += "Submitted by: Tushar Joshi | May 2026\n"

with open('/content/drive/MyDrive/kritter_project/README.md', 'w') as f:
    f.write(readme_content)

print("README saved ✅")
print("\nSUBMISSION CHECKLIST:")
print("✅ Top 100 ranked dataset")
print("✅ Interactive map")
print("✅ 3 charts")
print("✅ 7 slide presentation")
print("✅ README")
print("✅ Notebook")
print("\n🚀 READY TO SUBMIT!")

README saved ✅

SUBMISSION CHECKLIST:
✅ Top 100 ranked dataset
✅ Interactive map
✅ 3 charts
✅ 7 slide presentation
✅ README
✅ Notebook

🚀 READY TO SUBMIT!
